# 8 · Collect Linear Probing Results
Reads the `results-linear.csv` files produced by notebook 7 and assembles summary tables.

- **`df_ue_train`** — train and eval from the pre-training split.
- **`df_ue_train_on_eval`** — train from pre-training split, eval from held-out split.
- **`accuracy_dict`** — per-experiment accuracy array from a pre-computed `.npy` file.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
CKP_PATH = './metadata/DINOv2_checkpoint/'

EVAL_DIRS_UE_TRAIN = (
    'Symmetry_ue_train',
    'ProteinClass_ue_train',
    'MolecularWeight_kDa_ue_train',
    'ImagePixelSize_ue_train',
    'EMD_Resolution_ue_train',
    'DefocusV_ue_train',
)

EVAL_DIRS_TRAIN_ON_EVAL = (
    'Symmetry_ue_train_on_eval',
    'ProteinClass_ue_train_on_eval',
    'MolecularWeight_kDa_ue_train_on_eval',
    'ImagePixelSize_ue_train_on_eval',
    'EMD_Resolution_ue_train_on_eval',
    'DefocusV_ue_train_on_eval',
)

# Task names aligned with the per-experiment accuracy .npy axis order
TASK_NAMES = ('Symmetry', 'Function', 'Molecular_Weight',
              'Image_Pixel_Size', 'EMDB_Resolution', 'Defocus')

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Helper

In [4]:
def load_results(ckp_path, eval_dirs):
    """
    Read results-linear.csv for each task directory.
    Missing or unparseable files produce NaN so they are visible as gaps
    and are excluded from the mean automatically.
    """
    df = pd.DataFrame(index=sorted(eval_dirs), columns=['value'], dtype=float)
    for d in eval_dirs:
        csv_path = os.path.join(ckp_path, f'Linear_{d}', 'results-linear.csv')
        try:
            df.loc[d, 'value'] = pd.read_csv(csv_path).iloc[0, 0]
        except FileNotFoundError:
            print(f'  [missing]     {csv_path}')
        except (ValueError, IndexError) as exc:
            print(f'  [parse error] {csv_path}: {exc}')
    df.loc['mean'] = df['value'].mean()
    return df

## 1 · `ue_train` results

In [5]:
df_ue_train = load_results(CKP_PATH, EVAL_DIRS_UE_TRAIN)
df_ue_train

,value
DefocusV_ue_train,79.470003
EMD_Resolution_ue_train,96.844643
ImagePixelSize_ue_train,98.667902
MolecularWeight_kDa_ue_train,98.022795
ProteinClass_ue_train,97.816253
Symmetry_ue_train,99.010855
mean,94.972075


## 2 · `ue_train_on_eval` results

In [6]:
df_ue_train_on_eval = load_results(CKP_PATH, EVAL_DIRS_TRAIN_ON_EVAL)
df_ue_train_on_eval

,value
DefocusV_ue_train_on_eval,57.699376
EMD_Resolution_ue_train_on_eval,35.808334
ImagePixelSize_ue_train_on_eval,47.025657
MolecularWeight_kDa_ue_train_on_eval,22.724999
ProteinClass_ue_train_on_eval,35.940278
Symmetry_ue_train_on_eval,41.459376
mean,40.109670


## 3 · Per-experiment accuracy array

In [7]:
ln_accuracy = np.load(os.path.join(CKP_PATH, 'linear_accuracy_per_eval_empiar.npy'))
ln_accuracy = ln_accuracy.astype(object)
ln_accuracy = np.where(ln_accuracy == '', np.nan, ln_accuracy)
ln_accuracy[:, :, 1] = ln_accuracy[:, :, 1].astype(float)

accuracy_dict = {task: ln_accuracy[i] for i, task in enumerate(TASK_NAMES)}
print('Loaded tasks:', list(accuracy_dict.keys()))

Loaded tasks: ['Symmetry', 'Function', 'Molecular_Weight', 'Image_Pixel_Size', 'EMDB_Resolution', 'Defocus']
